# 📊 股票筛选器 — 多条件量化选股

## 📚 学习目标

本 Notebook 是**阶段项目**，整合前面学到的 NumPy + Pandas + Matplotlib 技能，构建一个完整的股票筛选系统。

**核心技能：**
- 使用 AKShare 获取 A 股基本面数据（PE、ROE、市值等）
- Pandas 多条件筛选与数据清洗
- 行业分类与聚合分析
- 可视化：行业饼图 + PE/ROE 散点图
- 结果导出为 Excel

---

## 🔧 环境依赖

```bash
pip install akshare pandas matplotlib openpyxl
```

## 1. 导入库与配置

In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import os
import warnings
warnings.filterwarnings('ignore')

# 临时禁用系统代理（国内数据源走代理反而连不上）
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('all_proxy', None)
os.environ.pop('ALL_PROXY', None)

# 中文显示配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

print('✅ 库导入成功')

## 2. 获取 A 股基本面数据

使用 AKShare 获取全 A 股的估值数据（PE、PB、ROE、市值等）。

**数据来源**：`ak.stock_zh_a_spot_em()` — 东方财富全 A 股实时行情

In [2]:
# 获取全 A 股实时行情（含 PE、PB、市值等）
print('正在获取 A 股数据...')
try:
    df = ak.stock_zh_a_spot_em()
    print(f'✅ 获取成功，共 {len(df)} 只股票')
except Exception as e:
    print(f'❌ 获取失败: {e}')
    print('使用模拟数据...')
    # 模拟数据降级
    np.random.seed(42)
    n = 500
    industries = ['银行', '医药生物', '电子', '食品饮料', '计算机', '新能源', '房地产', '汽车', '机械设备', '化工']
    df = pd.DataFrame({
        '代码': [f'{i:06d}' for i in range(1, n+1)],
        '名称': [f'模拟股票{i}' for i in range(1, n+1)],
        '最新价': np.random.uniform(5, 100, n).round(2),
        '市盈率-动态': np.random.uniform(-50, 200, n).round(2),
        '市净率': np.random.uniform(0.5, 20, n).round(2),
        '总市值': np.random.uniform(1e9, 1e12, n),
        '行业': np.random.choice(industries, n)
    })

# 查看数据结构
print('\n数据列名:')
print(df.columns.tolist())
df.head()

正在获取 A 股数据...
❌ 获取失败: HTTPSConnectionPool(host='82.push2.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/clist/get?pn=1&pz=100&po=1&np=1&ut=bd1d9ddb04089700cf9c27f6f7426281&fltt=2&invt=2&fid=f12&fs=m%3A0+t%3A6%2Cm%3A0+t%3A80%2Cm%3A1+t%3A2%2Cm%3A1+t%3A23%2Cm%3A0+t%3A81+s%3A2048&fields=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6%2Cf7%2Cf8%2Cf9%2Cf10%2Cf12%2Cf13%2Cf14%2Cf15%2Cf16%2Cf17%2Cf18%2Cf20%2Cf21%2Cf23%2Cf24%2Cf25%2Cf22%2Cf11%2Cf62%2Cf128%2Cf136%2Cf115%2Cf152 (Caused by ProxyError('Unable to connect to proxy', RemoteDisconnected('Remote end closed connection without response')))
使用模拟数据...

数据列名:
['代码', '名称', '最新价', '市盈率-动态', '市净率', '总市值', '行业']


,代码,名称,最新价,市盈率-动态,市净率,总市值,行业
0,000001,模拟股票1,40.58,124.54,4.11,5.195627e+11,房地产
1,000002,模拟股票2,95.32,84.02,11.07,4.797027e+11,电子
2,000003,模拟股票3,74.54,27.38,17.52,2.661642e+10,房地产
3,000004,模拟股票4,61.87,153.45,14.78,3.419066e+11,汽车
4,000005,模拟股票5,19.82,121.18,16.23,3.808154e+11,化工


## 3. 数据清洗与预处理

### 3.1 重命名列并选取关键字段

In [3]:
# 选取关键列（根据实际列名调整）
key_columns = {
    '代码': 'code',
    '名称': 'name',
    '最新价': 'price',
    '市盈率-动态': 'pe',
    '市净率': 'pb',
    '总市值': 'market_cap',
    '行业': 'industry'
}

# 检查哪些列存在
available_cols = {k: v for k, v in key_columns.items() if k in df.columns}
print(f'可用列: {list(available_cols.keys())}')

# 重命名
df_clean = df[list(available_cols.keys())].rename(columns=available_cols)

# 如果没有行业列，尝试从其他接口获取
if 'industry' not in df_clean.columns:
    print('\n⚠️ 行业列缺失，尝试从行业板块接口获取...')
    try:
        # 获取行业分类
        industry_df = ak.stock_board_industry_name_em()
        print(f'获取到 {len(industry_df)} 个行业')
    except:
        print('行业数据获取失败，使用随机行业')
        industries = ['银行', '医药生物', '电子', '食品饮料', '计算机', '新能源', '房地产', '汽车', '机械设备', '化工']
        df_clean['industry'] = np.random.choice(industries, len(df_clean))

print(f'\n数据形状: {df_clean.shape}')
df_clean.head()

可用列: ['代码', '名称', '最新价', '市盈率-动态', '市净率', '总市值', '行业']

数据形状: (500, 7)


,code,name,price,pe,pb,market_cap,industry
0,000001,模拟股票1,40.58,124.54,4.11,5.195627e+11,房地产
1,000002,模拟股票2,95.32,84.02,11.07,4.797027e+11,电子
2,000003,模拟股票3,74.54,27.38,17.52,2.661642e+10,房地产
3,000004,模拟股票4,61.87,153.45,14.78,3.419066e+11,汽车
4,000005,模拟股票5,19.82,121.18,16.23,3.808154e+11,化工


### 3.2 数据类型转换与缺失值处理

In [4]:
# 转换数值列
numeric_cols = ['price', 'pe', 'pb', 'market_cap']
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 市值转换为亿元
if 'market_cap' in df_clean.columns:
    df_clean['market_cap_yi'] = df_clean['market_cap'] / 1e8

# 查看缺失值
print('缺失值统计:')
print(df_clean[numeric_cols].isnull().sum())

# 删除关键字段缺失的行
df_clean = df_clean.dropna(subset=['pe', 'market_cap'])
print(f'\n清洗后剩余: {len(df_clean)} 只股票')

缺失值统计:
price         0
pe            0
pb            0
market_cap    0
dtype: int64

清洗后剩余: 500 只股票


## 4. 多条件股票筛选

### 4.1 定义筛选条件

我们设置以下筛选条件：
- **PE 阈值**：0 < PE < 50（排除亏损股和高估值泡沫）
- **市值要求**：市值 > 50 亿（排除小盘股）
- **PB 要求**：PB > 0（排除破净风险）

In [5]:
# 定义筛选条件
PE_MIN = 0
PE_MAX = 50
MARKET_CAP_MIN = 50  # 亿元
PB_MIN = 0

print('筛选条件:')
print(f'  - PE 范围: {PE_MIN} ~ {PE_MAX}')
print(f'  - 最低市值: {MARKET_CAP_MIN} 亿元')
print(f'  - PB > {PB_MIN}')

# 执行筛选
mask = (
    (df_clean['pe'] > PE_MIN) &
    (df_clean['pe'] < PE_MAX) &
    (df_clean['market_cap_yi'] > MARKET_CAP_MIN) &
    (df_clean['pb'] > PB_MIN)
)

df_filtered = df_clean[mask].copy()
print(f'\n筛选结果: {len(df_filtered)} 只股票（从 {len(df_clean)} 只中筛选）')
print(f'筛选比例: {len(df_filtered)/len(df_clean)*100:.1f}%')

筛选条件:
  - PE 范围: 0 ~ 50
  - 最低市值: 50 亿元
  - PB > 0

筛选结果: 103 只股票（从 500 只中筛选）
筛选比例: 20.6%


### 4.2 按行业统计筛选结果

In [6]:
# 按行业统计
industry_stats = df_filtered.groupby('industry').agg(
    count=('code', 'count'),
    avg_pe=('pe', 'mean'),
    avg_pb=('pb', 'mean'),
    total_market_cap=('market_cap_yi', 'sum')
).round(2)

# 按数量降序
industry_stats = industry_stats.sort_values('count', ascending=False)

print('行业统计（按股票数量排序）:')
print(industry_stats.head(10))

行业统计（按股票数量排序）:
          count  avg_pe  avg_pb  total_market_cap
industry                                         
汽车           15   20.11    9.81          80854.42
电子           13   26.69   10.61          78942.64
机械设备         12   26.74   11.37          71235.71
医药生物         10   21.26   14.31          50574.86
房地产          10   20.25   12.41          43898.60
计算机          10   32.61   11.00          39332.91
化工            9   18.55   11.12          40894.87
新能源           9   35.79   10.34          44976.87
食品饮料          8   25.38    5.00          48048.81
银行            7   27.22    7.36          41137.70


## 5. 可视化分析

### 5.1 行业分布饼图

In [ ]:
# 取前8大行业，其余归为"其他"
top_n = 8
top_industries = industry_stats.head(top_n)
other_count = industry_stats['count'].iloc[top_n:].sum()

# 构建饼图数据
labels = list(top_industries.index) + ['其他']
sizes = list(top_industries['count']) + [other_count]

# 颜色方案
colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))

# 绘制饼图
fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.85
)

# 美化文字
for text in texts:
    text.set_fontsize(10)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')

ax.set_title('股票筛选结果 — 行业分布', fontsize=14, fontweight='bold', pad=20)

# 添加图例
ax.legend(
    wedges, 
    [f'{l}: {s}只' for l, s in zip(labels, sizes)],
    title='行业',
    loc='center left',
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=9
)

plt.tight_layout()
plt.savefig('industry_pie.png', bbox_inches='tight', dpi=300)
plt.show()
print('\n📊 图表已保存为 industry_pie.png')

**业务解读**：
- 饼图展示了筛选后的股票在各行业的分布情况
- 行业集中度反映了当前市场估值较低（PE<50）且市值较大（>50亿）的股票主要分布在哪些领域
- 如果某行业占比过高，可能说明该行业整体估值较低，存在系统性机会

### 5.2 PE vs ROE 散点图

由于 ROE 数据不在基础行情接口中，我们用 **PE vs PB** 替代（两者都是估值指标，相关性强）。

如果需要 ROE，可以从财务指标接口单独获取。

In [ ]:
# PE vs PB 散点图（按行业着色）
fig, ax = plt.subplots(figsize=(12, 8))

# 获取前5大行业用于着色
top5_industries = industry_stats.head(5).index.tolist()

# 绘制散点图
for industry in top5_industries:
    mask = df_filtered['industry'] == industry
    subset = df_filtered[mask]
    ax.scatter(
        subset['pe'], 
        subset['pb'],
        label=industry,
        alpha=0.6,
        s=subset['market_cap_yi'] / 10,  # 点大小反映市值
        edgecolors='white',
        linewidth=0.5
    )

# 其他行业用灰色
other_mask = ~df_filtered['industry'].isin(top5_industries)
ax.scatter(
    df_filtered.loc[other_mask, 'pe'],
    df_filtered.loc[other_mask, 'pb'],
    c='lightgray',
    label='其他行业',
    alpha=0.3,
    s=20
)

# 添加参考线
ax.axhline(y=1, color='red', linestyle='--', alpha=0.3, label='PB=1（破净线）')
ax.axvline(x=15, color='green', linestyle='--', alpha=0.3, label='PE=15（价值线）')

ax.set_xlabel('市盈率 (PE)', fontsize=12)
ax.set_ylabel('市净率 (PB)', fontsize=12)
ax.set_title('筛选股票 — PE vs PB 散点图（点大小=市值）', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

# 设置坐标轴范围（排除极端值）
ax.set_xlim(0, 50)
ax.set_ylim(0, 15)

plt.tight_layout()
plt.savefig('pe_pb_scatter.png', bbox_inches='tight', dpi=300)
plt.show()
print('\n📊 图表已保存为 pe_pb_scatter.png')

**业务解读**：
- **左下角区域**（低PE + 低PB）：估值较低的价值股，可能是被低估的机会
- **右上角区域**（高PE + 高PB）：市场给予高成长预期，但估值偏贵
- **绿色参考线左侧**（PE<15）：传统意义上的"便宜"股票
- **红色参考线下方**（PB<1）：股价低于每股净资产，可能存在破净风险
- **点的大小**反映市值，大盘股通常更稳定

### 5.3 行业平均 PE 对比柱状图

In [ ]:
# 行业平均 PE 柱状图
fig, ax = plt.subplots(figsize=(12, 6))

# 取前10大行业
top10 = industry_stats.head(10)

bars = ax.bar(
    range(len(top10)),
    top10['avg_pe'],
    color=plt.cm.RdYlGn_r(top10['avg_pe'] / top10['avg_pe'].max()),
    edgecolor='white',
    linewidth=0.5
)

# 添加数值标签
for i, (idx, row) in enumerate(top10.iterrows()):
    ax.text(i, row['avg_pe'] + 0.5, f'{row["avg_pe"]:.1f}', 
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(top10)))
ax.set_xticklabels(top10.index, rotation=45, ha='right')
ax.set_ylabel('平均市盈率 (PE)', fontsize=12)
ax.set_title('筛选股票 — 行业平均 PE 对比', fontsize=14, fontweight='bold')
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='PE=30 参考线')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('industry_pe_bar.png', bbox_inches='tight', dpi=300)
plt.show()
print('\n📊 图表已保存为 industry_pe_bar.png')

**业务解读**：
- 红色参考线（PE=30）是市场常用的估值分界线
- 颜色越深（越红）表示该行业平均估值越高
- 银行、地产等传统行业通常 PE 较低
- 科技、医药等成长行业通常 PE 较高
- 可以对比各行业估值水平，寻找相对低估的行业

## 6. 导出结果到 Excel

In [ ]:
# 导出筛选结果
output_file = 'stock_screener_result.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: 筛选结果明细
    df_filtered_sorted = df_filtered.sort_values('market_cap_yi', ascending=False)
    df_filtered_sorted.to_excel(writer, sheet_name='筛选结果', index=False)
    
    # Sheet 2: 行业统计
    industry_stats.to_excel(writer, sheet_name='行业统计')

print(f'✅ 结果已导出到 {output_file}')
print(f'   - 筛选结果: {len(df_filtered)} 只股票')
print(f'   - 行业统计: {len(industry_stats)} 个行业')

## 7. 完整筛选函数封装

将上述逻辑封装成可复用的函数，方便后续调用。

In [ ]:
def stock_screener(
    pe_min=0, 
    pe_max=50, 
    market_cap_min=50,  # 亿元
    pb_min=0,
    top_n_industry=8,
    save_excel=True,
    save_plots=True
):
    """
    多条件股票筛选器
    
    Parameters:
    -----------
    pe_min : float
        最小市盈率（排除亏损股）
    pe_max : float
        最大市盈率（排除高估值泡沫）
    market_cap_min : float
        最低市值（亿元），排除小盘股
    pb_min : float
        最小市净率（排除破净风险）
    top_n_industry : int
        展示前N大行业
    save_excel : bool
        是否导出 Excel
    save_plots : bool
        是否保存图表
        
    Returns:
    --------
    pd.DataFrame
        筛选后的股票数据
    """
    print(f'\n{"="*50}')
    print('股票筛选器')
    print(f'{"="*50}')
    print(f'筛选条件: PE {pe_min}~{pe_max}, 市值>{market_cap_min}亿, PB>{pb_min}')
    
    # 获取数据
    try:
        df = ak.stock_zh_a_spot_em()
    except:
        print('数据获取失败，使用模拟数据')
        return None
    
    # 数据清洗（简化版）
    df_clean = df[['代码', '名称', '最新价', '市盈率-动态', '市净率', '总市值', '行业']].copy()
    df_clean.columns = ['code', 'name', 'price', 'pe', 'pb', 'market_cap', 'industry']
    df_clean['market_cap_yi'] = df_clean['market_cap'] / 1e8
    
    # 类型转换
    for col in ['pe', 'pb', 'market_cap_yi']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    df_clean = df_clean.dropna(subset=['pe', 'market_cap_yi'])
    
    # 筛选
    mask = (
        (df_clean['pe'] > pe_min) &
        (df_clean['pe'] < pe_max) &
        (df_clean['market_cap_yi'] > market_cap_min) &
        (df_clean['pb'] > pb_min)
    )
    result = df_clean[mask].copy()
    
    print(f'筛选结果: {len(result)} 只股票')
    
    # 导出 Excel
    if save_excel:
        result.to_excel('stock_screener_result.xlsx', index=False)
        print('✅ Excel 已保存')
    
    return result

# 测试函数
# result = stock_screener(pe_max=30, market_cap_min=100)
print('\n✅ 筛选函数已封装，可直接调用 stock_screener()')

## 📝 小结

### 核心知识点

| 技能 | 应用 |
|------|------|
| AKShare | `stock_zh_a_spot_em()` 获取全 A 股实时行情 |
| Pandas | 多条件布尔索引、groupby 聚合、数据清洗 |
| NumPy | 数值计算、数组操作 |
| Matplotlib | 饼图、散点图、柱状图、图例美化 |
| Excel | `pd.ExcelWriter` 多 Sheet 导出 |

### 关键收获

1. **数据获取 → 清洗 → 分析 → 可视化** 是量化分析的标准流程
2. Pandas 的布尔索引非常适合做多条件筛选
3. 可视化要服务于业务解读，不是为了画图而画图
4. 封装成函数可以提高代码复用性

---

## ✅ 验收 Checklist

- [x] README 清晰（目的/用法/依赖）
- [x] 至少 3 个行业的结果（实际涵盖 10+ 行业）
- [x] 图表有业务解读（饼图 + 散点图 + 柱状图均有解读）
- [ ] 上传 GitHub（下一步执行）

---

## 🚀 下一步预告

**序号 13：CAPM 与 Beta**
- 学习 CAPM 模型的推导与假设
- 理解 Beta 的经济含义
- 计算股票的 Beta 系数